In [1]:
import vald_api_utilities as vau
import vald_smartspeed_api as vssa

%load_ext autoreload
%autoreload 2

# Initialize organizational parameters
Vald should provide a "Client ID", "Client Secret", and a "Tenant ID". The "Tenant ID" is some what interchangable with the "Team ID" and can be acquired through the External Tenant API call.

In [2]:
# Read credentials using vald_api_utilities function
creds = vau.read_credentials()

client_id = creds['client_id']
client_secret = creds['client_secret']
tenantId = creds['tenant_id']

print(f"Client ID: {client_id}")
print(f"Client Secret: {client_secret}")
print(f"Tenant ID: {tenantId}")

Client ID: 
Client Secret: 
Tenant ID: 


### Initialize Vald API Class Function  
The class functions as a wrapper for the External Tenant and External Profile API calls.

In [3]:
vald = vau.vald_api(client_id,client_secret,tenant_id=tenantId)

### Step 1 - Get Token
"get_token()" should always be your first api call as this gives you a temparoray access key to be used for all future calls

In [4]:
vald.get_token()

#### OPTIONAL - "get_all_tenants()" or "get_tenant_info()"
Use "get_all_tenants()" to return a dictionary with all tenants.
Use "get_tenant_info()" to return a dictionary with additional inforamtion about specific tenant.

In [5]:
vald.get_all_tenants()
vald.tenants

[{'id': '61476d2d-917a-4b2e-b2f6-e43346d2cb66',
  'name': 'University of Oregon'}]

In [6]:
# If not initializing class function with "tenant ID" then enter 
# tenant_id here, using commented example.
# vald.get_tenant_info(tenant_id=vald.tenants[0]['id'])

# If initialized class function with "Tenant ID" then use the following.
vald.get_tenant_info()
vald.tenant_info

{'id': '61476d2d-917a-4b2e-b2f6-e43346d2cb66',
 'name': 'University of Oregon',
 'sport': 'MultiSportCollege',
 'league': 'NCAA',
 'logoUri': 'https://content.valdperformance.com/logos/61476d2d-917a-4b2e-b2f6-e43346d2cb66.jpg'}

### Step 2 - Get information on the categories and groups
Using the specified "Tenant ID", get information on the categories and groups for the "Tenant ID".

In [7]:
# Generated dataframe containing ID and name of each category
vald.get_tenant_categories()
vald.categories_df

,id,syncId,name
0,306f33af-2939-480b-bbd7-2ce643c888f0,None,Uncategorised
1,3a32a047-1292-49f1-a8aa-8f830ea58ac8,None,Position
2,1fdaf750-970a-4047-badc-945e6e6994cf,None,Team
3,2828d5d2-ce26-4297-b4c1-a16039889a86,None,Sport


In [9]:
# Generated dataframe of groups ID, name, and category ID. 
vald.get_tenant_groups()
# vald.groups_df.head()

### Step 3 - Get all profiles in a specific group.
Use "get_group_profiles()" to generate dataframe of all athlete profiles in a given group based on the category and group dataframe. It is important to generate both the group and category dateframes as both of these will used to identify the correct IDs.

In [19]:
# categoru_name is used to get the category ID which is used to get the correct group ID.
# This insures that when there are groups' with the same name in different categories, 
# the correct group ID is found.

# Replace 'Baseball' (group name) with desired value.
vald.get_group_profiles('Track and Field',category_name='Team')
# vald.profile_df.head()

# Smart Sppeed API Call
After gathering the tenant ID and profile IDs we can use this information to pull force decks results.

In [20]:
# Initialize Force Decks class function with tenant ID and header from Vald class function.
smartspeed = vssa.smartspeed_api(vald.tenant_id,vald.header,region='USA')

### Step 1 - Get a collection of tests
Using "get_multiple_tests()", you need to include a start and stop date for the period you'd like to pull tests from. You can also provide a modified date and or a profile ID to filter on. If no modified data is given then the start date will be used for the modified date. If no profile ID is given then that will not be included in API call.

Start and stop date should be within 6 months of each other

In [26]:
start_date = '14/01/2025' # date must be in "dd/mm/yyyy" format
stop_date = '16/01/2025' # date must be in "dd/mm/yyyy" format
profileID = vald.profile_df.loc[6,'profileId']

smartspeed.get_tests_results(start_date,stop_date,profileID=profileID)
smartspeed.tests_df.head()

,id,testResultId,groupUnderTestId,profileId,testDateUtc,deviceCount,repCount,testTypeName,testName,isValid,...,reactiveDelayMinimumInSeconds,reactiveDelayMaximumInSeconds,events,durationInSeconds,lapCount,intervalType,testStandardType,dropHeight,dropHeightEnabled,weightKg
0,ddc07ed7-750d-40bf-89cd-50a5447437b0,8d5645dd-4190-4215-8482-25ece1c449df,None,6dac6c2d-b628-4681-a1cf-0987cbe70ea7,2025-01-15T22:54:25,4,1,OneWay,30 m WSOC,True,...,0.0,0.0,None,None,1,None,Standard,0.0,False,None
1,40da1ed2-a204-44a5-a33b-93807ddbe38a,a9de110a-8436-4fbd-8106-46675c67d07b,None,6dac6c2d-b628-4681-a1cf-0987cbe70ea7,2025-01-15T22:42:47,4,1,OneWay,30 m WSOC,True,...,0.0,0.0,None,None,1,None,Standard,0.0,False,None
